<h1 style="text-align:center;">Optimal Execution<br></h1>
<div style="text-align:center;">Ariel Kalingking</div>
<div style="text-align:center;">akalingking@gmail.com</div>
<p style="text-align:center;">Appendix Python Code</p>

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import logging

imagepath = "../paper/figures/"
matplotlib.use('TkAgg')
logging.getLogger().setLevel(logging.ERROR)
figsize = (6,5)
fontbig = 8
fontmed = 7
fontsmall = 6

plt_inline_enable = False
if plt_inline_enable:
    %matplotlib inline

In [2]:
### Model parameters ###
eta = 0.1                  # Risk aversion for trading (eta > 0)
gamma = 0.01               # Risk aversion for inventory (gamma > 0)
sigma = 0.05               # Volatility of the price process (sigma > 0)
T = 1.0                    # Time horizon (e.g., 1 day)

# Time Discretization
N_t = 200                  # Number of time steps (backward)
t_min, t_max = 0, T        # Boundary condition for time
dt = T / N_t               # Time step size

# Inventory Discretization
N_q = 100                   # Number of inventory grid points
q_min, q_max = -500, 500    # Boundary condition for inventory size
dq = (q_max - q_min) / N_q  # Inventory step size

In [3]:
### Analytical Solution for V(q,t) ###
# Calculates V(q,t) using the closed form solution
# V(q, t) = sqrt(gamma * sigma^2 * eta) * tanh(sqrt(gamma * sigma^2 * eta) (T-t)) * q^2
def value_function(q, t, eta, gamma, sigma, T):
    k_val = np.sqrt(gamma * sigma**2 * eta)
    time_remaining = np.maximum(0, T - t)
    argument = (k_val * time_remaining) / eta    
    v_qt = k_val * np.tanh(argument) * (q**2)
    return v_qt

## Construct the grid dataset for plotting ##
q_values = np.linspace(q_min, q_max, N_q)
t_values = np.linspace(t_min, t_max, N_t)
T_mesh, Q_mesh = np.meshgrid(t_values, q_values)
vq_values = value_function(Q_mesh, T_mesh, eta, gamma, sigma, T)
Z = vq_values
assert np.shape(Z)[0] == np.shape(T_mesh)[0]
assert np.shape(Z)[1] == np.shape(Q_mesh)[1]


# Plot the V(q,t) using the sampled boundary q_max, q_min and t_max, t_min
fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111, projection='3d')
surface = ax.plot_surface(T_mesh, Q_mesh, Z, cmap='viridis', edgecolor='none')
ax.set_xlabel('Time, $\Delta t$', fontsize=fontmed)
ax.set_ylabel('Inventory, $q$', fontsize=fontmed)
ax.set_zlabel('$V(q,t)$', fontsize=fontmed)
ax.set_title('Value Function $V(q,t)$\n$Analytical$', fontsize=fontbig, fontweight="bold", y=.99)
ax.tick_params(axis='both', labelcolor="black", labelsize=fontsmall)
ax.invert_xaxis()
ax.set_box_aspect(aspect=None, zoom=0.87)
plt.tight_layout()
plt.savefig(imagepath+"/value_function_analytical.pdf")
if plt_inline_enable:
    plt.show()

In [4]:
#### Plot optimal trading tate, (analytical) ####
def optimal_trading_rate_analytical(q, t, eta, gamma, sigma, T):
    # Calculate v*(q,t) from the derivation
    time_remaining = np.maximum(0, T - t)
    first_term = q * np.sqrt( (gamma*sigma**2) / eta ) 
    second_term = np.tanh( sigma * np.sqrt(gamma / eta) * time_remaining ) 
    v_star = first_term * second_term
    return v_star

# Create a meshgrid for q and t for analytical calculation
q_values = np.linspace(q_min, q_max, N_q)
t_values = np.linspace(t_min, t_max, N_t)

T_mesh, Q_mesh = np.meshgrid(t_values, q_values)
v_star_values = optimal_trading_rate_analytical(Q_mesh, T_mesh, eta, gamma, sigma, T)
Z = v_star_values
assert np.shape(Z)[0] == np.shape(T_mesh)[0]
assert np.shape(Z)[1] == np.shape(Q_mesh)[1]

# --- Create the 3D Surface Plot for Optimal Trading Rate ---
fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(T_mesh, Q_mesh, Z, cmap='viridis', edgecolor='none')
ax.set_xlabel('Time, $\Delta t$', fontsize=fontmed)
ax.set_ylabel('Inventory, $q$', fontsize=fontmed)
ax.set_zlabel('$v^*(q,t)$', fontsize=fontmed)
ax.set_title('Optimal Control $v^*(q,t)$\n$Analytical$', fontsize=fontbig, fontweight="bold", y=.99)
ax.tick_params(axis='both', labelcolor="black", labelsize=fontsmall)
ax.invert_xaxis()
ax.set_box_aspect(aspect=None, zoom=0.87)
plt.tight_layout()
plt.savefig(imagepath+"/control_function_analytical.pdf")
if plt_inline_enable:
    plt.show()